# Phase 1 — Reproducing "An Optimized Ensemble Model for Predicting Average Localization Error of WSNs"

**Base paper:** Nti, Rout & Yeboah, *Decision Analytics Journal* 12 (2024) 100510.
**What this notebook does:** loads the dataset, rebuilds the paper's experiment (Random Forest tuned by Grid Search, Random Search and Bayesian Optimization), and compares **our** numbers with **the paper's**.

**How to use it:** run the cells top to bottom (*Run All*). The tuning cell (section 5) takes a few minutes because Grid Search tries 400 combinations.
The same code also lives in `src/` and `scripts/`, so the notebook and `python run_all.py` give the same results (same seed).

> Rule for the report: only write "reproduced" for things this notebook actually ran. Paper numbers and our numbers are always shown side by side and labelled.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# make `src/` importable whether the notebook is opened from the repo root or from notebooks/
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.config import load_config, out_dir
from src.data import FEATURES, TARGET, load_data, split_data
from src.metrics import METRIC_NAMES
from src.paper_reference import (CORR_WITH_ALE, FEATURE_IMPORTANCE, OPTIMIZER_LABELS, TABLE1)
from src.pipeline import (comparison_table, evaluate_all, feature_importances, metrics_table,
                          params_comparison_table, save_phase1, select_by_cv, tune_all, tuning_table)
from src.plots import (plot_correlation, plot_distributions, plot_importance,
                       plot_metric_comparison, plot_pred_vs_actual)

pd.set_option("display.precision", 4)
cfg = load_config()
SEED = cfg["seed"]
np.random.seed(SEED)
FIG = out_dir(cfg, "figures")
print("Seed:", SEED)

## 1. The problem in plain words

A **wireless sensor network (WSN)** is a set of cheap sensors scattered over an area (a forest, a farm, a disaster zone).
A sensor's reading is only useful if you know **where** the sensor is. GPS on every sensor is too costly, so only a few
sensors, the **anchors**, know their position, and the rest estimate theirs from the anchors.
That estimate is never perfect. The average gap between estimated and true position is the **Average Localization Error (ALE)**, in meters. Lower is better.

ALE depends on four network settings (our input **features**):

| feature | meaning |
|---|---|
| `anchor_ratio` | share of sensors that know their exact position |
| `trans_range` | how far each sensor's radio signal reaches (m) |
| `node_density` | how many sensors are deployed |
| `iterations` | rounds the localization algorithm runs |

**The paper's idea:** instead of re-running a slow localization simulation for every candidate setup, train a
**Random Forest** on past runs so it can **predict ALE instantly** from the four settings. Random Forests have hyperparameters
(number of trees, depth, ...) that must be tuned, and the paper compares three tuning methods to see which gives the best predictions.

**So what are we testing?** (1) Can a tuned Random Forest predict ALE accurately? (2) Does the paper's claim hold that
**Random Search** is the best tuner? We implement it ourselves and compare honestly.

## 2. Load and inspect the data

Source: UCI ML Repository, dataset 844 (Singh et al., 2020), 107 rows. The loader keeps the 4 features + ALE and **drops the extra "std of ALE" column** (it describes the target, so using it would leak the answer).
If the file is missing, run `python scripts/00_download_data.py` first (see `data/README.md`).

In [ ]:
df = load_data(cfg)
print("Shape:", df.shape, "  (paper: 107 rows, 4 features + target)")
print("Missing values:", int(df.isna().sum().sum()), "  Duplicate rows:", int(df.duplicated().sum()))
display(df.head())
desc = df.describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
display(desc)

### Is this the same dataset the paper used?
We compare our summary statistics with the paper's Table 1. If (almost) everything matches, we are using the same data.

In [ ]:
rows = []
for col, stats in TABLE1.items():
    for stat, paper_val in stats.items():
        ours = float(desc.loc[col, stat])
        rows.append({"variable": col, "statistic": stat, "paper": paper_val, "ours": round(ours, 4),
                     "match": bool(np.isclose(ours, paper_val, rtol=0.01, atol=0.011))})
chk = pd.DataFrame(rows)
print(f"{int(chk['match'].sum())}/{len(chk)} statistics match the paper's Table 1")
display(chk[~chk["match"]] if (~chk["match"]).any() else "All statistics match.")

In [ ]:
fig1 = plot_distributions(df, FIG / "eda_distributions.png")
fig2 = plot_correlation(df, FIG / "eda_correlation.png")
corr = df.corr()[TARGET].drop(TARGET)
display(pd.DataFrame({"paper": pd.Series(CORR_WITH_ALE), "ours": corr}).loc[FEATURES])
plt.show()

**Reading the correlations:** a negative number means "more of this feature -> lower error". Node density (about -0.65) and iterations (about -0.40) are the strongest drivers, which makes physical sense: more sensors and more refinement rounds give better position estimates.

## 3. Train/test split and preprocessing

Like the paper: **80% train / 20% test**, Min-Max scaling. Two details matter:
* The scaler is fitted **inside the training data only** (in a `Pipeline`), so no information leaks from the test set.
* Random Forests don't care about feature scale, so scaling doesn't change the predictions. We keep it only to follow the paper.

With 107 rows the test set has only ~22 rows, so single-split results are **noisy**. Keep that in mind when comparing with the paper.

In [ ]:
X_train, X_test, y_train, y_test = split_data(df, cfg, SEED)
print(f"train: {len(X_train)} rows, test: {len(X_test)} rows")

## 4. Baseline: Random Forest with default settings

Always start with an untuned model. If tuning barely beats the baseline, that is an important finding too.

In [ ]:
from src.pipeline import evaluate_params
base_metrics, base_model, base_pred = evaluate_params({}, X_train, y_train, X_test, y_test, SEED)
display(pd.DataFrame([base_metrics], index=["Baseline RF"]))

## 5. Hyperparameter tuning: Grid vs Random vs Bayesian

We tune four Random Forest settings (`n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`) using **5-fold cross-validation on the training set only**.

* **Grid Search** tries every combination from a fixed grid (400 here). Thorough but slow.
* **Random Search** tries a fixed number of random combinations (50 here). Fast.
* **Bayesian Optimization** (Optuna's TPE) also gets 50 trials, but each one is chosen using what the earlier trials showed. Smart but has overhead.

Random and Bayesian get the **same budget** (50 trials) so the comparison is fair. This cell takes a few minutes.

In [ ]:
tuned = tune_all(X_train, y_train, cfg, SEED)
display(tuning_table(tuned))
print("Best optimizer by cross-validation score:", OPTIMIZER_LABELS[select_by_cv(tuned)])

## 6. Test-set results

Each tuned Random Forest is refitted on the full training set and scored **once** on the held-out test set.
**MAPE** is shown as a fraction (0.14 = 14%), as in the paper.

In [ ]:
metrics, models, preds = evaluate_all(tuned, X_train, y_train, X_test, y_test, SEED)
display(metrics_table(metrics))
plot_pred_vs_actual(y_test.to_numpy(), preds, metrics, FIG / "pred_vs_actual.png")
plt.show()

Points close to the dashed diagonal mean the prediction is close to the true ALE.

## 7. Which features matter most?

Random Forest feature importance, shown next to the paper's Fig. 5. We use the optimizer chosen by **cross-validation** (never by test score, which would leak).

In [ ]:
imp = feature_importances(models)
selected = select_by_cv(tuned)
display(imp.loc[list(FEATURE_IMPORTANCE)].assign(paper=pd.Series(FEATURE_IMPORTANCE)))
plot_importance(imp, selected, FIG / "feature_importance.png")
plt.show()

## 8. Comparison with the original paper

Grey = reported by the paper, coloured = produced by this notebook.

**Caveats to quote in your report:**
* The paper's reported MSE and RMSE are not consistent with each other (e.g. Grid Search: RMSE 0.238 implies MSE about 0.057, but 0.046 is reported), so compare mainly on RMSE, MAE, R2 and MAPE.
* The paper does not report a random seed, the exact grid, or the search budgets, so exact numbers cannot be matched.
* The paper lists identical tuning times for Grid and Random Search (52.74 s), which looks like a copy error.

In [ ]:
comp = comparison_table(metrics)
display(comp)
display(params_comparison_table(tuned))
plot_metric_comparison(comp, FIG / "paper_vs_ours_metrics.png")
plt.show()

## 9. Your discussion (write this part yourself)

Use these prompts for the report's *Results* and *Discussion* sections. Replace this cell with your own sentences.

* Are our R2/RMSE close to the paper's? If not, which of these could explain it: different split or seed, different grid or budget, library versions, a different copy of the dataset?
* Does our ranking of the three optimizers match the paper's (Random Search best, then Grid, then Bayesian)? Remember the test set has only ~22 rows.
* Did tuning beat the untuned baseline by a meaningful margin?
* Do our most important features match the paper's (iterations, node density, then anchor ratio and transmission range)?
* Which findings are solid, and which might just be noise from one split?

## 10. (Recommended) How stable is this? Repeat with different splits

Run `python scripts/04_seed_robustness.py` once. It repeats split -> tune -> test with several seeds. The cell below shows its results if the file exists.
This tells you whether "Random Search is best" survives a different random split, which matters a lot with 107 rows.

In [ ]:
p = out_dir(cfg, "tables") / "seed_robustness_summary.csv"
if p.exists():
    display(pd.read_csv(p, header=[0, 1], index_col=0))
    from IPython.display import Image
    display(Image(filename=str(FIG / "seed_robustness.png")))
else:
    print("No robustness results yet. Run: python scripts/04_seed_robustness.py")

## 11. Save everything to `results/`

Writes the tables and a JSON summary used by the report and by `scripts/03_compare_with_paper.py`.

In [ ]:
res = {
    "seed": SEED, "n_train": len(X_train), "n_test": len(X_test),
    "tuned": tuned, "metrics": metrics, "importances": imp,
    "selected_by_cv": selected, "y_test": y_test.to_numpy(),
    "preds": {k: np.asarray(v) for k, v in preds.items()},
}
print("Saved to:", save_phase1(res, cfg))

## 12. What to put in the Phase 1 report

| Report section | Where the material comes from |
|---|---|
| Dataset + preprocessing | sections 2-3, `eda_distributions.png`, `eda_correlation.png`, Table 1 check |
| Methodology + experimental setup | sections 3-5 and `configs/config.yaml` (seed, split, CV, search space, budgets) |
| Results | section 6, `test_metrics.csv`, `pred_vs_actual.png`, `feature_importance.png` |
| Comparison with the paper | section 8, `paper_vs_ours_metrics.csv`, `paper_vs_ours_hyperparameters.csv` |
| Discussion | section 9 prompts + the seed-robustness table (section 10) |